In [5]:
import pathlib
from argparse import ArgumentParser
import yaml
import torch
import torchmetrics


In [6]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
import src.audio_transforms as at


In [3]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [8]:
audio_config = config['data']['audio']


## Sanity Checking

Put RMS before combine to test if level influences model 

In [9]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.001),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

In [12]:
model = AttentionalTrackingModule(config)
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)

In [13]:
ckpt_path = '../attn_cue_models/attn_cue_jsin_pilot_no_pretrain_bs_64_lr_1e-4/checkpoints/epoch=1-step=120790.ckpt'

ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])

<All keys matched successfully>

In [14]:
fg_acc = torchmetrics.Accuracy()
bg_acc = torchmetrics.Accuracy()

In [15]:
model = model.eval().cuda()
fg_acc.cuda()
bg_acc.cuda()

Accuracy()

In [16]:
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

In [17]:
from tqdm import tqdm

In [18]:
for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


100%|██████████| 1050/1050 [06:46<00:00,  2.58it/s]


In [19]:
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

Foreground accuracy: 0.8032825589179993
Background accuracy: 0.8199334144592285


## Increase RMS

In [20]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.2),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

In [21]:
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

In [22]:
fg_acc.reset()
bg_acc.reset()


In [23]:
for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


100%|██████████| 1050/1050 [06:47<00:00,  2.58it/s]


In [24]:
# With louder sounds
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

Foreground accuracy: 0.8130351901054382
Background accuracy: 0.8070884943008423


In [13]:
# f_word

tensor([639, 497, 391, 580], device='cuda:0')

In [17]:
# fg_out.log_softmax(-1).argmax(-1)

tensor([700, 497, 391, 580], device='cuda:0')

In [14]:
# b_word

tensor([604, 250, 623, 324], device='cuda:0')

In [20]:
# bg_out.log_softmax(-1).argmax(-1)

tensor([605, 250, 623, 324], device='cuda:0')

## Eval at same RMS as trained 

In [25]:


audio_config = config['data']['audio']
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

In [26]:
no_sep_dataet = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)

In [27]:
no_sep_val_loader = torch.utils.data.DataLoader(no_sep_dataet, batch_size=4, num_workers=10)

In [28]:
fg_acc.reset()
bg_acc.reset()


In [29]:
for ix, batch in tqdm(enumerate(no_sep_val_loader), total=len(no_sep_val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    
#     if ix % 100 == 0:
#         print(f"Foreground accuracy: {f_acc}")
#         print(f"Background accuracy: {b_acc}")
        
    if ix == len(no_sep_val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


100%|██████████| 1050/1050 [06:47<00:00,  2.58it/s]


In [30]:
# Final performance
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

Foreground accuracy: 0.8142245411872864
Background accuracy: 0.8151760101318359


## Eval across SNR at low RMS

In [31]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.001),
            at.CombineWithRandomDBSNR(low_snr=-10, high_snr=10), # set to 0 so foreground/background at same level 
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

In [32]:
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

In [33]:
fg_acc.reset()
bg_acc.reset()

In [ ]:
!pip install 

In [34]:
for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    
#     print(fg_out.argmax())s
#     break


100%|██████████| 1050/1050 [06:46<00:00,  2.58it/s]


In [35]:
# Final performance
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

Foreground accuracy: 0.7956708073616028
Background accuracy: 0.8004281520843506


## Eval at low SNR

In [36]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.001),
            at.CombineWithRandomDBSNR(low_snr=-10, high_snr=-5), # set to 0 so foreground/background at same level 
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

fg_acc.reset()
bg_acc.reset()


for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    

# Final performance
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

100%|██████████| 1050/1050 [06:46<00:00,  2.58it/s]


Foreground accuracy: 0.7186013460159302
Background accuracy: 0.8349190950393677


## Eval at high SNR

In [37]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.001),
            at.CombineWithRandomDBSNR(low_snr=5, high_snr=10), # set to 0 so foreground/background at same level 
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config)
        ])

dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms)
val_loader = torch.utils.data.DataLoader(dataset, batch_size=4, num_workers=10)

fg_acc.reset()
bg_acc.reset()


for ix, batch in tqdm(enumerate(val_loader), total=len(val_loader)//4):
    batch = [item.cuda() for item in batch]
    mix, f_cue, b_cue, f_word, b_word = batch
    fg_out = model(f_cue, mix)
    bg_out = model(b_cue, mix)
    
    fg_acc(fg_out, f_word)
    bg_acc(bg_out, b_word)
    if ix == len(val_loader)//4:
        break
    

# Final performance
#
print(f"Foreground accuracy: {fg_acc.compute()}")
print(f"Background accuracy: {bg_acc.compute()}")

100%|██████████| 1050/1050 [06:47<00:00,  2.58it/s]


Foreground accuracy: 0.8392007350921631
Background accuracy: 0.7076593637466431
